# 12. generation_utils.py: от prompt до RGB по функциям

Эта лабораторная исследует **все 10 функций** закреплённого generation_utils.py. Сначала читаем контракт, затем предсказываем результат, выполняем небольшой опыт и проверяем исходник. Цель — понимать порядок операций, формы тензоров, CFG, sampling, conditioning и offload.

Содержание: 1) get_sparse_params; 2) adaptive_mean_std_normalization; 3) normalize_first_frame; 4) get_velocity; 5) generate; 6) resize_video; 7) encode_video; 8) generate_sample; 9) generate_sample_ti2i; 10) generate_sample_i2v.

Перед началом полезны лабораторные 01, 02, 04, 06 и 07. Большие веса не нужны: small tensor probes отделяют контракты от качества генерации.


### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


## Как именно запускаем исходные функции

Loader извлекает AST FunctionDef из сохранённого файла и выполняет **неизменённые тела функций** в отдельном namespace. Реальны PyTorch, torchvision resize/crop и fast_sta_nabla. Вызовы DiT/VAE/text encoder заменяются маленькими **записывающими заглушками**, чтобы видеть параметры и порядок вызовов.

Чистые helpers и generate исполняются с настоящим torch на CPU. Для трёх orchestrators нужен `cpu_cuda=True`: отдельный facade переводит hardcoded CUDA Generator в CPU Generator, CUDA autocast в nullcontext и empty_cache в записываемый no-op. BF16 noise сохраняется. Это явный **учебный adapter**, не доказательство поддержки CPU/MPS исходным pipeline. Глобальный torch и сторонние пакеты не изменяются. Quiet tqdm отключает только отображение progress bar.

Заглушки не содержат pretrained weights, реальных tokenizer или codec. Их скорость и изображения не измеряют Kandinsky latency/quality. Реальный CUDA запуск остаётся в лабораторной 10.

В Colab начните с новой среды выполнения и выполните setup сверху: открытая ранее сессия может хранить старую копию labkit без generation_lab.py. После обновления файла notebook сам по себе уже загруженный код не обновляется.


In [ ]:
from labkit.generation_lab import load_generation_functions, FakeDiT, FakeVAE, FakeTextEmbedder, make_generation_conf
from types import SimpleNamespace
import ast,inspect,warnings
original_generator=torch.Generator
gu=load_generation_functions()
adapter_events=[]
gu_cpu=load_generation_functions(cpu_cuda=True,adapter_log=adapter_events)
assert torch.Generator is original_generator
function_nodes=[n for n in ast.parse((SOURCE/'kandinsky/generation_utils.py').read_text()).body if isinstance(n,ast.FunctionDef)]
assert set(gu['_function_names'])=={n.name for n in function_nodes} and len(function_nodes)==10
for n in function_nodes:
    print(f'{n.name:34} lines {n.lineno}–{n.end_lineno}  {inspect.signature(gu[n.name])}')
print('Pinned source:',REVISION)


## Карта потока и соглашение о формах

```mermaid
flowchart LR
    P[Caption + negative caption] --> TE[Text embedder]
    S[Seed] --> N[Gaussian noise THWC]
    TE --> G[get_velocity]
    N --> LOOP[generate: shifted grid + Euler]
    SP[get_sparse_params] --> LOOP
    LOOP --> G
    G --> LOOP
    LOOP --> D[Unscale + BCTHW + VAE decode]
    D --> U[clamp + uint8 RGB]
    I[Input RGB image] --> R[resize_video]
    R --> E[encode_video for TI2I]
    E --> LOOP
    IF[Scaled image latent for I2V] --> LOOP
    LOOP --> NF[normalize_first_frame for I2V]
    NF --> D
```

| Обозначение | Форма | Значение |
|---|---|---|
| Codec RGB input | B,C,T,H,W | Channel-first pixels |
| Wrapper shape | B,T,h,w,Cz | Задание размеров noise в latent space |
| Core sampler img | T,h,w,Cz | Batch=1 в настоящих callers; channels-last |
| DiT input с visual_cond | T,h,w,2Cz+1 | current state, zero visual channels, mask |
| Decode input | B,Cz,T,h,w | После деления на VAE scaling_factor |
| T2V/I2V return | B,3,Tpx,Hpx,Wpx uint8 | RGB tensor; MP4 сохраняет внешний pipeline |
| T2I/TI2I image return | B,3,Hpx,Wpx uint8 | При image_vae=True time axis удаляется |

`shape` допускает B синтаксически, но callers этой ревизии используют B=1: flatten B*T, RoPE длины T и один caption не образуют рабочий batch video pipeline. Не выводите поддержку batch>1 из одной сигнатуры.


## 1. get_sparse_params: подготовка NABLA mask

Контракт: `conf`, `batch_embeds['visual']` формы **T,h,w,C**, device → dict sparse_params или None. Название batch_embeds не означает наличие batch axis.

1. Проверяется temporal patch size=1, даже если attention не NABLA.
2. Размеры latent делятся на patch sizes: получается grid DiT tokens.
3. Для NABLA grid делится на spatial blocks 8×8 token positions; fast_sta_nabla строит boolean adjacency соседних blocks.
4. Dictionary сообщает attention engine mask, fractal layout, visual_shape, threshold P и параметры локального окна. Другие attention types дают None.

Mask имеет форму 1×1×Nblocks×Nblocks, не Ntokens². `P` — параметр последующего NABLA selection, не число шагов Euler. Для выбранного patch2 H/W latent должны делиться на 16 при block attention; floor division в этой функции не проверяет корректность дальнейшего fractal layout. Здесь проверяем только подготовку mask, не запускаем sparse GPU kernel.

Предскажите число blocks при T=3, h=w=32 и patch=(1,2,2).


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def get_sparse_params',28)
mask_conf=make_generation_conf(); mask_conf.model.dit_params.patch_size=(1,2,2)
mask_input=torch.zeros(3,32,32,2)
assert gu['get_sparse_params'](mask_conf,{'visual':mask_input},'cpu') is None
mask_conf.model.attention.type='nabla'
mask_conf.model.attention.wT=3; mask_conf.model.attention.wH=1; mask_conf.model.attention.wW=1
sparse=gu['get_sparse_params'](mask_conf,{'visual':mask_input},'cpu')
assert sparse['visual_shape']==(3,16,16)
assert sparse['sta_mask'].shape==(1,1,12,12)
assert sparse['sta_mask'][0,0].diagonal().all()
print({k:v for k,v in sparse.items() if k!='sta_mask'})
plt.figure(figsize=(4,4)); plt.imshow(sparse['sta_mask'][0,0],cmap='gray',vmin=0,vmax=1)
plt.xlabel('key block'); plt.ylabel('query block'); plt.title('3 time × 2 × 2 spatial blocks'); plt.show()


## 2. adaptive_mean_std_normalization: ограниченная поправка статистик

Source имеет форму T,H,W,C. Mean и std считаются **отдельно для каждого source frame**, по осям H/W/C. Reference mean/std считаются **по всему reference tensor**, включая время. Стандартное torch.std использует correction=1; важно согласовать это при проверке результата.

Пусть source frame имеет mean m и std s. Целевые статистики ограничены:

$$m^*=\mathrm{clip}(m_{ref},m-0.05,m+0.1),\qquad s^*=\mathrm{clip}(s_{ref},s-0.1,s+0.25).$$
$$y=(x-m)s^*/s+m^*.$$

Получаем ограниченное изменение контраста/смещения латента. Эта операция не обязана полностью приблизить source к reference. Constants 0.05/0.1/0.25 — эвристика конкретной реализации, не VAE prior или Flow Matching loss. Input не меняется inplace. При s=0 отсутствует epsilon, поэтому constant source даёт NaN.


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def adaptive_mean_std_normalization',18)
norm_rng=torch.Generator().manual_seed(1202)
norm_source=torch.randn(2,4,4,2,generator=norm_rng)*torch.tensor([0.7,1.2])[:,None,None,None]
norm_source+=torch.tensor([-0.5,0.5])[:,None,None,None]
norm_reference=torch.randn(5,4,4,2,generator=norm_rng)*0.5+1.5
norm_before=norm_source.clone()
norm_output=gu['adaptive_mean_std_normalization'](norm_source,norm_reference)
source_m=norm_source.mean((1,2,3)); source_s=norm_source.std((1,2,3))
expected_m=torch.clamp(norm_reference.mean(),source_m-0.05,source_m+0.1)
expected_s=torch.clamp(norm_reference.std(),source_s-0.1,source_s+0.25)
assert torch.allclose(norm_output.mean((1,2,3)),expected_m,atol=1e-6)
assert torch.allclose(norm_output.std((1,2,3)),expected_s,atol=1e-6)
assert torch.equal(norm_source,norm_before)
print('source means/std:',source_m.tolist(),source_s.tolist())
print('global reference mean/std:',norm_reference.mean().item(),norm_reference.std().item())
print('output means/std:',expected_m.tolist(),expected_s.tolist())
constant_output=gu['adaptive_mean_std_normalization'](torch.ones(2,4,4,2),norm_reference)
assert not torch.isfinite(constant_output).all()
print('Constant source produces nonfinite values:',not torch.isfinite(constant_output).all().item())


## 3. normalize_first_frame: первые четыре latent frames

Функция clone-ит input, выбирает `samples[:4]` и reference slice начиная с index4 (по умолчанию до index9). Применяет предыдущую нормализацию; `clump_values=True` дополнительно ограничивает результат global min/max reference. Кадры с index4 и далее сохраняются.

Это postprocessing latent перед decoder в I2V; четыре latent frames не равны четырём RGB кадрам из-за temporal compression. Название singular first_frame скрывает изменение четырёх кадров.

**Граничные случаи исходника:** T=1 возвращает tuple `(latents, message)`, хотя обычная ветка возвращает Tensor. При T=2…4 reference пуст и возникают NaNs; с clump_values=True min/max пустого tensor вызывает ошибку. Для обычной ветки нужны T≥5, достаточный sample count, конечные statistics и ненулевая source std. Reference std может быть равна нулю. Показываем это как исследование контракта, не исправляем vendored source.

Число четыре само по себе не является доказанной ошибкой: это явное поведение исходника, намерение автора не документировано. Обычный Python I2V 5s имеет T=31; T2V Distill эту функцию вообще не вызывает. Подробный аудит main, достижимости edge cases и return-type conflict находится в [лабораторной 13](13_normalization_and_vae_contract_audit.ipynb).


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def normalize_first_frame',23)
frames=torch.randn(8,4,4,2,generator=norm_rng)
frames[:4]+=0.4
frames_before=frames.clone()
frames_normal=gu['normalize_first_frame'](frames)
frames_clamped=gu['normalize_first_frame'](frames,clump_values=True)
assert torch.equal(frames,frames_before)
assert torch.equal(frames_normal[4:],frames[4:])
assert frames_clamped[:4].min()>=frames[4:].min() and frames_clamped[:4].max()<=frames[4:].max()
plt.plot(frames.mean((1,2,3)), 'o-', label='input')
plt.plot(frames_normal.mean((1,2,3)), 'x--',label='first 4 normalized')
plt.axvline(3.5,color='gray',linestyle=':'); plt.xlabel('latent frame index'); plt.ylabel('mean'); plt.legend(); plt.show()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for length in (1,2,3,4,5,6):
        candidate=torch.randn(length,4,4,2,generator=norm_rng)
        result=gu['normalize_first_frame'](candidate)
        if length==1:
            assert isinstance(result,tuple) and result[0] is candidate
            print('T=1: tuple, original tensor returned')
        else:
            finite=bool(torch.isfinite(result).all()); assert finite==(length>=5)
            print('T=',length,'finite:',finite)
    try: gu['normalize_first_frame'](torch.randn(3,4,4,2),clump_values=True)
    except RuntimeError as error: print('Expected empty-reference clamp failure:',type(error).__name__)


## 4. get_velocity: один DiT forward или CFG из двух

Эта функция не двигает latent: она получает velocity нужного размера. `x` — текущий model input, `t` — обычно tensor shape[1] из диапазона [0,1]. В DiT поступает **t×1000**, text token states, pooled embedding, RoPE positions, mask и `conf.metrics.scale_factor`. Последний параметр масштабирует RoPE по трём осям: time, height, width. В выбранном config временной множитель равен 1. Это масштаб позиционных координат; scheduler_scale отдельно меняет временную сетку интегратора.

Сначала всегда считается conditional ветка. Если |guidance_weight−1|>1e−6, дополнительно считается null/negative ветка:

$$v=v_{null}+w(v_{cond}-v_{null}).$$

w=1 означает conditional generation с одним DiT forward, а не отсутствие текста. w=0 возвращает null prediction, но исходник всё равно сначала вычисляет conditional — итого два forward. `@torch.no_grad` отключает autograd: training loss находится вне этого inference helper.

Записывающий ProbeVelocity ниже возвращает известные числа для двух conditioning. Это позволяет проверить формулу, время и masks точно.


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def get_velocity',42)
class ProbeVelocity:
    def __init__(self,visual_cond=False): self.visual_cond=visual_cond; self.calls=[]
    def __call__(self,x,text,pooled,time,visual_rope,text_rope,**kwargs):
        self.calls.append({'x':x.clone(),'time':time.clone(),'mask':kwargs.get('attention_mask'),
                           'scale_factor':kwargs.get('scale_factor'),'text_rope':text_rope.clone()})
        return x[...,:2]*0 + pooled.mean()  # без no_grad эта формула сохраняла бы связь с x
cond={'text_embeds':torch.ones(1,3,4),'pooled_embed':torch.tensor([[2.]])}
null={'text_embeds':torch.zeros(1,2,4),'pooled_embed':torch.tensor([[-1.]])}
positions=[torch.arange(2)]*3; cond_pos=torch.arange(3); null_pos=torch.arange(2)
cond_mask=torch.tensor([[True,True,False]]); null_mask=torch.tensor([[True,False]])
core_conf=make_generation_conf()
velocity_input=torch.randn(2,2,2,2,requires_grad=True)
for weight,expected,ncalls in [(0.,-1.,2),(1.,2.,1),(2.,5.,2)]:
    probe=ProbeVelocity()
    output=gu['get_velocity'](probe,velocity_input,torch.tensor([0.5]),cond,null,
        positions,cond_pos,null_pos,weight,core_conf,attention_mask=cond_mask,null_attention_mask=null_mask)
    assert torch.allclose(output,torch.full_like(output,expected)) and not output.requires_grad
    assert len(probe.calls)==ncalls and probe.calls[0]['time'].item()==500
    assert probe.calls[0]['mask'] is cond_mask
    if ncalls==2: assert probe.calls[1]['mask'] is null_mask
    print('guidance:',weight,'velocity:',output.mean().item(),'DiT forwards:',len(probe.calls))


## 5. generate: shifted grid, Euler и conditioning

Core sampler принимает уже созданный noise img. Он не генерирует случайные числа и не вызывает VAE или text encoder.

1. get_sparse_params строит static mask до возможного tensor-parallel split.
2. Grid t=1→0 сдвигается $f_s(t)=st/(1+(s-1)t)$.
3. На каждом шаге собирается model input, затем get_velocity.
4. Обновляются только первые C_out каналов: `img[..., :C_out] += delta_t * velocity`.
5. Возвращаются первые C_out channels как view изменённого img.

Delta_t отрицателен. При постоянной velocity весь update равен −velocity независимо от shift: sum(delta_t)=−1. `seed` здесь не используется; seed влияет на wrapper noise. `progress` также не используется для выбора tqdm. Исходник не проверяет num_steps>0 и scheduler_scale>0.

### 5.1. Проверяем время, NFE, мутацию и лишние каналы

Extra editing channels остаются неизменными в памяти, но не попадают в return. Для w=1,N=16 получается16 DiT forwards; для обычного CFG с w≠1 —32. Это подсчёт вызовов, не замер latency.


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def generate(',60)
def run_core(state,model,steps=4,scale=1.,first=None,seed=1,tp_mesh=None):
    rope=[torch.arange(state.shape[0]),torch.arange(state.shape[1]),torch.arange(state.shape[2])]
    return gu['generate'](model,'cpu',state,steps,cond,null,rope,cond_pos,null_pos,1.,scale,first,
                          core_conf,progress=False,seed=seed,tp_mesh=tp_mesh,
                          attention_mask=cond_mask,null_attention_mask=null_mask)
initial=torch.randn(6,2,2,2,generator=torch.Generator().manual_seed(1205))
fig,axes=plt.subplots(1,2,figsize=(10,3))
for scale in (1.,5.):
    state=initial.clone(); probe=ProbeVelocity(); result=run_core(state,probe,scale=scale)
    assert torch.allclose(result,initial-2.,atol=1e-6)
    assert result.data_ptr()==state.data_ptr() and not torch.equal(state,initial)
    times=torch.tensor([c['time'].item()/1000 for c in probe.calls]+[0.])
    assert torch.allclose(times,shifted_schedule(4,scale))
    axes[0].plot(times,'o-',label=f'scale={scale:g}')
    axes[1].plot(torch.diff(times),'o-',label=f'scale={scale:g}')
axes[0].set(xlabel='grid index',ylabel='t'); axes[1].set(xlabel='step',ylabel='negative delta_t')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()
assert torch.equal(run_core(initial.clone(),ProbeVelocity(),seed=1),run_core(initial.clone(),ProbeVelocity(),seed=999))
extra=torch.randn(6,2,2,3)
editing_state=torch.cat([initial,extra],-1); editing_before=editing_state.clone()
trimmed=run_core(editing_state,ProbeVelocity())
assert trimmed.shape[-1]==2 and torch.equal(editing_state[...,2:],editing_before[...,2:])
try: run_core(initial.clone(),ProbeVelocity(),steps=0)
except UnboundLocalError: print('Expected num_steps=0 failure: pred_velocity is never assigned')


### 5.2. I2V mask: условие попадает в current state

При model.visual_cond=True input имеет [img, visual_cond, mask]. В этой ревизии visual_cond создаётся нулевым и остаётся нулевым. Если first_frames передан, **img[:1]** заменяется условием и mask[:1]=1. Поэтому первая часть input содержит condition, а отдельные visual_cond channels — нули.

Перед каждым forward первый frame закрепляется заново, но после forward Euler update снова его сдвигает. Wrapper I2V отдельно закрепляет frame после sampling, затем применяет normalize_first_frame. Проверим core поведение на простом поле.


In [ ]:
first_latent=torch.full((1,2,2,2),0.7)
conditioned_probe=ProbeVelocity(visual_cond=True)
conditioned_result=run_core(initial.clone(),conditioned_probe,first=first_latent)
for call in conditioned_probe.calls:
    model_input=call['x']
    assert model_input.shape[-1]==5
    assert torch.allclose(model_input[:1,...,:2],first_latent)
    assert torch.equal(model_input[...,2:4],torch.zeros_like(model_input[...,2:4]))
    assert model_input[:1,...,4:].eq(1).all() and model_input[1:,...,4:].eq(0).all()
assert not torch.allclose(conditioned_result[:1],first_latent)
print('First frame after final Euler update:',conditioned_result[:1].mean().item(),'condition:',first_latent.mean().item())


### 5.3. Tensor parallel: место разделения и ограничения проверки

Если tp_mesh задан и first_frames=None, img делится по **height dim1**, а не по time. get_sparse_params вызывается до split. generate_sample собирает shards через all_gather и cat(dim1), generate_sample_i2v такого gather не делает. Для реального distributed запуска нужны parallelized DiT, согласованные RoPE/masks и одинаковые shard shapes; простой torch.chunk не реализует collective communication.

Ниже выполняем исходную ветку split с имитацией rank1/world2 и constant ProbeVelocity. Проверяем только индекс разделения. Математика настоящего parallel DiT разобрана в лабораторной08.


In [ ]:
class MeshSliceProbe:
    def get_local_rank(self): return 1
    def size(self): return 2
shard_input=torch.randn(6,4,2,2)
shard_output=run_core(shard_input.clone(),ProbeVelocity(),tp_mesh={'tensor_parallel':MeshSliceProbe()})
assert shard_output.shape==(6,2,2,2)
assert torch.allclose(shard_output,shard_input[:,2:]-2.,atol=1e-6)
print('global THWC:',shard_input.shape,'rank1 THWC:',shard_output.shape)


### 5.4. Подключаем настоящий tiny DiT вместо ProbeVelocity

Лабораторная06 создаёт уменьшенные классы Kandinsky DiT. Подключим их к **настоящему generate**: теперь velocity зависит от x, времени и текстовых тензоров, а не задана константой. labkit.native использует отдельный CPU adapter для DiT (eager FP32 и SDPA); weights случайные. Проверяем интерфейс inference, формы и NFE, не качество видео.

Вход sampler T×4×4×16 превращается в T×4×4×33 через visual_cond; DiT возвращает16 velocity channels. Text/Qwen размеры уменьшены до32, pooled/CLIP до16.


In [ ]:
from labkit.native import tiny_dit
tiny_model=tiny_dit('cpu')
tiny_conf=make_generation_conf(patch_size=(1,2,2))
tiny_state=torch.randn(2,4,4,16)
tiny_cond={'text_embeds':torch.randn(1,6,32),'pooled_embed':torch.randn(1,16)}
tiny_null={key:torch.zeros_like(value) for key,value in tiny_cond.items()}
tiny_rope=[torch.arange(2),torch.arange(2),torch.arange(2)]
tiny_calls=[]
handle=tiny_model.register_forward_hook(lambda module,args,out: tiny_calls.append(tuple(out.shape)))
tiny_result=gu['generate'](tiny_model,'cpu',tiny_state.clone(),2,tiny_cond,tiny_null,
    tiny_rope,torch.arange(6),torch.arange(6),1.,5.,None,tiny_conf,
    attention_mask=torch.ones(1,6,dtype=torch.bool),null_attention_mask=torch.ones(1,6,dtype=torch.bool))
handle.remove()
assert tiny_result.shape==tiny_state.shape and torch.isfinite(tiny_result).all()
assert len(tiny_calls)==2 and all(shape==(2,4,4,16) for shape in tiny_calls)
print('Actual tiny DiT parameters:',sum(p.numel() for p in tiny_model.parameters()),'sampler forwards:',len(tiny_calls))
del tiny_model; release()


## 6. resize_video: cover resize и центральный crop

Функция сохраняет aspect ratio, увеличивая/уменьшая image до полного покрытия target rectangle, затем обрезает центр. Это **не letterbox**: поля не добавляются, часть краёв теряется.

Input torchvision tensor здесь имеет C,H,W или T,C,H,W; TI2I caller передаёт B=1,C,H,W. B,C,T,H,W codec layout не является допустимой автоматической video-обработкой этого helper. `visual_size=(target_height,target_width)`.

На примере H=12,W=20,target8×8 scale=min(12/8,20/8)=1.5; resize даёт8×13 (int округляет вниз), затем crop8×8. Нечётная разница ширин делает left/right crop различным на один пиксель. Геометрия одна для всех frames.


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def resize_video',17)
y_coord=torch.linspace(0,1,12)[:,None].expand(12,20)
x_coord=torch.linspace(0,1,20)[None,:].expand(12,20)
coordinate_image=torch.stack([x_coord,y_coord,((x_coord>0.4)&(x_coord<0.6)).float()])
coordinate_video=torch.stack([coordinate_image,coordinate_image.flip(-1)])
resized=gu['resize_video'](coordinate_video,(8,8))
assert resized.shape==(2,3,8,8)
fig,axes=plt.subplots(1,2,figsize=(8,3))
axes[0].imshow(coordinate_image.permute(1,2,0)); axes[0].set_title('12×20 input')
axes[1].imshow(resized[0].permute(1,2,0)); axes[1].set_title('cover + center crop 8×8')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()


## 7. encode_video: layout/scaling и два codec контракта

Input B,C,T,H,W → output B,T,h,w,Cz. Image branch требует T=1: удаляет time axis, берёт `.latent_dist.sample()` и возвращает time axis. Video branch берёт `vae.encode(data)[0]`, предполагая Tensor. Затем latent умножается inplace на scaling_factor и permute переносит channels в конец. Return может быть non-contiguous view.

**Проверенная несовместимость:** Hunyuan vae.py этой ревизии возвращает AutoencoderKLOutput с DiagonalGaussianDistribution. Поэтому `[0]` — posterior object, а не latent Tensor, и video branch падает на умножении, если encode успешно завершился. I2V caller обходит этот helper: отдельный get_first_frame_from_image использует `.latent_dist.sample()`, что соответствует posterior API. Обычный I2I caller выбирает image_vae=True; T2V не вызывает encode_video. Это helper/API mismatch, не доказательство отказа этих стандартных маршрутов. [Лабораторная 13](13_normalization_and_vae_contract_audit.ipynb) воспроизводит проблему на настоящем уменьшенном VAE encode и отделяет её от CPU CUDA-memory-planner ограничения.

Первый опыт ниже исследует ожидаемый Tensor-returning contract заглушки. Второй воспроизводит несовместимость на настоящих diffusers posterior/output classes без загрузки VAE weights. Явная формула исправленного caller — `posterior.sample()` или `posterior.mode()` до scaling; выбор должен соответствовать training/conditioning контракту. Vendored helper не меняем.


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def encode_video',10)
codec_input=torch.randn(1,3,9,16,16)
for image_branch in (False,True):
    encode_log=[]; codec=FakeVAE(encode_log,image_vae=image_branch,channels=2,scaling_factor=0.5)
    pixels=codec_input[:,:,:1] if image_branch else codec_input
    encoded=gu['encode_video'](pixels,codec,image_vae=image_branch)
    assert encoded.shape==(1,1 if image_branch else 3,2,2,2)
    unscaled=encoded.permute(0,4,1,2,3)/codec.config.scaling_factor
    assert torch.allclose(encoded, (unscaled*codec.config.scaling_factor).permute(0,2,3,4,1))
    print('image_vae:',image_branch,'BCTHW:',pixels.shape,'→ BTHWC:',encoded.shape,'contiguous:',encoded.is_contiguous())
from diffusers.models.autoencoders.vae import DiagonalGaussianDistribution
from diffusers.models.modeling_outputs import AutoencoderKLOutput
class PosteriorReturningVAE:
    config=SimpleNamespace(scaling_factor=0.5)
    def encode(self,pixels):
        return AutoencoderKLOutput(latent_dist=DiagonalGaussianDistribution(torch.zeros(1,4,3,2,2)))
try: gu['encode_video'](codec_input,PosteriorReturningVAE(),image_vae=False)
except TypeError as error: print('Expected actual posterior-contract failure:',error)
posterior=PosteriorReturningVAE().encode(codec_input).latent_dist
safe_mode=(posterior.mode()*0.5).permute(0,2,3,4,1)
assert safe_mode.shape==(1,3,2,2,2)


## Три orchestration-функции: от компонентов к готовым кадрам

Мы уже изучили отдельные операции. Теперь разберём функции, которые соединяют их в pipeline:
`generate_sample` (T2V, также имеет режим duration=1), `generate_sample_ti2i` (генерация/редактирование изображения),
`generate_sample_i2v` (анимация заданного изображения).

**Что исполняется в опытах.** `load_generation_functions(cpu_cuda=True)` извлекает тела
функций из закреплённого `generation_utils.py` без переписывания алгоритма. В их отдельном
пространстве имён CUDA Generator заменяется CPU Generator, CUDA autocast — пустым контекстом,
а очистка CUDA cache — пустой операцией. Глобальный `torch` и исходный репозиторий не меняются.
Так мы можем проверить управляющую логику на вашем Mac; это **не проверка полного запуска
pretrained Kandinsky на MPS** и не обещание совпадения CPU/CUDA случайных чисел.

Текстовый энкодер, DiT и VAE ниже — маленькие **инструментированные заглушки**:
они запоминают входы и порядок вызовов. DiT возвращает постоянную velocity;
VAE выполняет простые pooling/upsampling. Графики показывают движение данных,
а не качество генерации. Их интерфейсы повторяют те методы, которые вызывает модуль.

| Договорённость | В этом опыте | В реальном видео Lite |
|---|---|---|
| Число объектов `B` | 1 | обёртки рассчитаны прежде всего на 1 |
| Латенты | `[T,H,W,C]=[6,2,2,2]` | например `[31,64,96,16]` для 512×768 |
| Упаковка входа sampler | `[B·T,H,W,C]` | batch и time объединены |
| Вход video VAE decoder | `[B,C,T,H,W]` | channels-first |
| Выход video decoder | `[B,3,4(T−1)+1,8H,8W]` | затем uint8 в диапазоне 0…255 |

Случай `B>1` нельзя объявить поддержанным только потому, что `shape` содержит `bs`:
caption передаётся как список **одного** текста, RoPE строится для `duration`, а I2V
фиксирует `img[:1]`. В лаборатории проверяем договорённости для `B=1`.


In [ ]:
from labkit.generation_lab import (
    load_generation_functions, FakeDiT, FakeVAE, FakeTextEmbedder, make_generation_conf,
)
generation = load_generation_functions(cpu_cuda=True)

def wrapper_components(*, visual_cond=False, instruct_type=None, image_vae=False):
    events = []
    dit = FakeDiT(events, channels=2, visual_cond=visual_cond,
                  instruct_type=instruct_type, velocity=0.25)
    vae = FakeVAE(events, image_vae=image_vae, scaling_factor=0.5)
    text = FakeTextEmbedder(events)
    return events, dit, vae, text

conf_wrapper = make_generation_conf()
video_shape = (1, 6, 2, 2, 2)
print('Загружены настоящие тела трёх wrapper-функций; компоненты — учебные.')


## 8. `generate_sample`: полный маршрут T2V

Прочитайте функцию как последовательность восьми шагов:

1. Из `shape=(B,T,H,W,C)` извлекаются размеры **латентного** видео. Создаётся BF16 шум
   `[B·T,H,W,C]`; `seed` используется здесь. Сам `generate` не создаёт шум заново.
2. `duration==1` выбирает текстовый шаблон `image`, иначе — `video`.
3. Положительный и отрицательный prompt кодируются **оба**, даже при `guidance_weight=1`.
   `cu_seqlens[-1]` задаёт число текстовых токенов для RoPE.
4. Embeddings переносятся на `device`; позиции visual RoPE строятся по патч-сетке.
   Spatial patch size делит `H,W`; по времени код использует `duration`.
5. `generate` интегрирует velocity от `t=1` до `t=0`. Если `visual_cond=True`, к `C`
   каналам шума добавляются `C` нулевых conditioning-каналов и **один** канал маски:
   вход DiT имеет `2C+1` каналов, выход velocity — `C`.
6. При tensor parallel выполняется `all_gather` по разрезанной высоте. В CPU опыте
   `tp_mesh=None`; распределённый запуск здесь не проверяется.
7. Латенты распаковываются в `[B,T,H,W,C]`, делятся на `vae.config.scaling_factor`,
   затем переставляются в `[B,C,T,H,W]` и декодируются.
8. Decoder output обрезается к `[-1,1]`, преобразуется в `[0,255]` и `uint8`.
   Функция **возвращает tensor**, не пишет MP4 и не задаёт fps.

`offload=True` добавляет переносы моделей: text → CPU; DiT → рабочее устройство → CPU;
VAE → устройство декодирования → CPU. Здесь это журнал вызовов, не измерение RAM/VRAM.
Параметр `text_embedder_device` внутри этой функции не используется для переноса энкодера:
его начальное размещение обеспечивает внешний pipeline.

В этой ревизии T2IPipeline делегирует I2IPipeline и вызывает generate_sample_ti2i с image=None, image_vae=True. Наличие image-ветки в generate_sample не означает, что именно её использует настоящий T2I caller.

**Отдельно проверьте устройства masks.** Wrapper переносит tensors из словаря embeddings
на устройство DiT, но не вызывает `.to(device)` для `attention_mask` и
`null_attention_mask`. Они могут остаться на устройстве текстового энкодера. При
размещении text encoder на CPU, а DiT на CUDA необходимо проверить и согласовать
устройства масок, особенно при padding. CPU опыт ниже проверяет передачу масок,
но не подтверждает совместимость такого смешанного размещения. Это относится
ко всем трём wrapper-функциям этой ревизии.


In [ ]:
source_excerpt('kandinsky/generation_utils.py', 'def generate_sample(', 30)
events_t2v, dit_t2v, vae_t2v, text_t2v = wrapper_components(visual_cond=True)
frames_t2v = generation['generate_sample'](
    video_shape, 'Учебный красный куб', dit_t2v, vae_t2v, conf_wrapper, text_t2v,
    num_steps=3, guidance_weight=1.0, scheduler_scale=1.0, seed=42,
    device='cpu', vae_device='cpu', text_embedder_device='cpu',
    progress=False, offload=True, tp_mesh=None,
)
assert frames_t2v.shape == (1, 3, 21, 16, 16)
assert frames_t2v.dtype == torch.uint8
assert len(dit_t2v.calls) == 3
assert len(text_t2v.calls) == 2  # negative prompt всё равно закодирован
assert vae_t2v.decode_calls[0].shape == (1, 2, 6, 2, 2)
for call in dit_t2v.calls:
    assert call['x'].shape == (6, 2, 2, 5)
    assert torch.count_nonzero(call['x'][..., 2:]) == 0
print('Выход:', tuple(frames_t2v.shape), frames_t2v.dtype)
print('Журнал:', *events_t2v, sep='\n')
plt.imshow(frames_t2v[0, :, 0].permute(1, 2, 0).numpy())
plt.title('Учебный decoder: первый кадр (не pretrained генерация)')
plt.axis('off'); plt.show()


### Опыт: где именно seed, Euler и VAE scale влияют на результат

Для постоянного поля $v=0.25$ сумма шагов времени равна $-1$, поэтому в точной арифметике
самpler сдвинул бы шум на $-0.25$. BF16 округляет каждый шаг; ниже вручную повторяем
**те же операции и тот же dtype**, затем сравниваем вход VAE, а не RGB после clipping.
Это сильнее, чем сравнить только форму: проверяем, что интеграция, деление на scale
и перестановка осей оказались на правильном месте.

Второе сравнение меняет только seed. Равный seed даёт равный tensor в **одном CPU backend**;
это не означает тождественные шумы на CUDA. Третье сравнение проверяет режим `duration=1`:
`generate_sample` сохраняет video API decoder, даже когда он получает один кадр.


In [ ]:
initial = torch.randn(6, 2, 2, 2,
    generator=torch.Generator(device='cpu').manual_seed(42), dtype=torch.bfloat16)
manual = initial.clone()
for dt in torch.diff(torch.linspace(1, 0, 4)):
    manual += dt * torch.full_like(manual, 0.25)
manual_decoder_input = (manual.reshape(1, 6, 2, 2, 2) / 0.5).permute(0, 4, 1, 2, 3)
assert torch.equal(manual_decoder_input, vae_t2v.decode_calls[0])

def repeat_t2v(seed, shape=video_shape):
    events, dit, vae, text = wrapper_components(visual_cond=True)
    result = generation['generate_sample'](
        shape, 'Учебный красный куб', dit, vae, conf_wrapper, text,
        num_steps=3, guidance_weight=1.0, scheduler_scale=1.0, seed=seed,
        device='cpu', vae_device='cpu', text_embedder_device='cpu', progress=False,
    )
    return result, vae, text

repeat, repeat_vae, _ = repeat_t2v(42)
other, other_vae, _ = repeat_t2v(43)
assert torch.equal(frames_t2v, repeat)
assert not torch.equal(repeat_vae.decode_calls[0], other_vae.decode_calls[0])
one_frame, _, one_text = repeat_t2v(42, (1, 1, 2, 2, 2))
assert one_text.calls[0]['type_of_content'] == 'image'
assert one_frame.shape == (1, 3, 1, 16, 16)
moves = [event[0] for event in events_t2v if event[0].endswith('.to')]
assert moves == ['text.to', 'dit.to', 'dit.to', 'vae.to', 'vae.to']
print('Seed воспроизводим; Euler + scale + permute совпали точно.')
print('Переносы моделей:', moves, '; duration=1 output:', tuple(one_frame.shape))


## 9. `generate_sample_ti2i`: два разных канала conditioning

Здесь `image` — уже **RGB tensor** `[B,3,H_rgb,W_rgb]` в диапазоне 0…255. Это другой
контракт, чем `images` в I2V, который мы изучим следующим. Для редактирования одного
изображения используем `duration=1`.

1. RGB сначала resized/cropped к `(8H,8W)`, где `H,W` — латентные размеры.
2. Если `dit.instruct_type=='channel'`, RGB приводится к BF16 и `[-1,1]`, затем кодируется
   `encode_video`. Получается scaled latent `[1,H,W,C]`.
3. К **шуму** добавляются `C` каналов закодированного изображения и одна mask=1:
   `[noise_C, edit_latent_C, mask_1]`. Если image отсутствует, оба дополнения нулевые.
   Это conditioning input, а не инициализация sampler этим изображением: старт остаётся шумом.
4. При наличии image текстовый режим становится `image_edit`, и resized RGB передаётся
   в `text_embedder.encode(..., images=image)` **для обоих prompt**.
   Поэтому image может влиять через visual часть Qwen и через DiT channel conditioning.
5. В `generate` Euler обновляет только первые `C` каналов velocity; добавленные edit
   channels не интегрируются. Перед decoder они удаляются.
6. При `image_vae=True` encoder использует `.latent_dist.sample()` и decoder получает
   `[B,C,H,W]`; при `False` сохраняется video API `[B,C,1,H,W]`.

В нашем опыте `visual_cond=False`: к каналу редактирования не добавляется ещё один
video-conditioning блок. Следует сверять обе настройки с конкретным DiT/config,
а не суммировать произвольные channel packs.


In [ ]:
source_excerpt('kandinsky/generation_utils.py', 'def generate_sample_ti2i(', 32)
gradient = torch.linspace(0, 255, 20 * 32).reshape(20, 32)
input_rgb = torch.stack([gradient, gradient.flip(-1),
                         torch.full_like(gradient, 127)], dim=0)[None].to(torch.uint8)
events_edit, dit_edit, vae_edit, text_edit = wrapper_components(
    instruct_type='channel', image_vae=True,
)
edited = generation['generate_sample_ti2i'](
    (1, 1, 2, 2, 2), 'Добавить красный цвет', dit_edit, vae_edit,
    conf_wrapper, text_edit, num_steps=2, guidance_weight=1.0, seed=42,
    device='cpu', vae_device='cpu', text_embedder_device='cpu',
    progress=False, offload=True, image_vae=True, image=input_rgb,
)
assert edited.shape == (1, 3, 16, 16)
assert vae_edit.encode_calls[0].shape == (1, 3, 16, 16)
assert vae_edit.decode_calls[0].shape == (1, 2, 2, 2)
expected_edit_latent = (vae_edit.encodings[0] * 0.5).permute(0, 2, 3, 1)
for call in dit_edit.calls:
    assert call['x'].shape == (1, 2, 2, 5)
    assert torch.equal(call['x'][..., 2:4], expected_edit_latent)
    assert torch.all(call['x'][..., 4] == 1)
assert torch.equal(dit_edit.calls[0]['x'][..., 2:], dit_edit.calls[-1]['x'][..., 2:])
assert all(call['type_of_content'] == 'image_edit' for call in text_edit.calls)
assert all(call['images'][0].shape == (1, 3, 16, 16) for call in text_edit.calls)
assert [event[0] for event in events_edit[:3]] == ['vae.to', 'vae.encode', 'vae.to']
print('RGB → encode:', tuple(vae_edit.encode_calls[0].shape))
print('DiT:', tuple(dit_edit.calls[0]['x'].shape), '; decode:', tuple(edited.shape))
fig, axs = plt.subplots(1, 2, figsize=(7, 3))
axs[0].imshow(input_rgb[0].permute(1, 2, 0).numpy()); axs[0].set_title('Вход RGB')
axs[1].imshow(edited[0].permute(1, 2, 0).numpy()); axs[1].set_title('Выход fake decoder')
for ax in axs: ax.axis('off')
plt.show()


### Опыт: image routing и отключение одного conditioning-пути

Сравним три режима, сохраняя общий шум/seed. Нас интересуют размеры и маршруты,
а не сходство изображений: наш постоянный DiT специально не читает edit-каналы.

| Режим | Вход DiT | `type_of_content` | VAE encode до sampling | RGB в text encoder |
|---|---|---|---|---|
| image + channel | `2C+1` | `image_edit` | да | да |
| без image + channel | `2C+1`, edit=0,mask=0 | `image` | нет | нет |
| image + без channel | `C` | `image_edit` | нет | да |

Последняя строка показывает, что отсутствие `instruct_type='channel'` убирает
**один** путь conditioning. RGB всё ещё передаётся multimodal text embedder.
В fake encoder это только журнал; реальная семантика зависит от его реализации.


In [ ]:
for label, input_image, instruct in [
    ('image + channel', input_rgb, 'channel'),
    ('no image + channel', None, 'channel'),
    ('image + no channel', input_rgb, None),
]:
    events, dit, vae, text = wrapper_components(instruct_type=instruct, image_vae=True)
    result = generation['generate_sample_ti2i'](
        (1, 1, 2, 2, 2), 'Пример', dit, vae, conf_wrapper, text,
        num_steps=1, guidance_weight=1.0, seed=42, device='cpu', vae_device='cpu',
        text_embedder_device='cpu', progress=False, image_vae=True, image=input_image,
    )
    expected_channels = 5 if instruct == 'channel' else 2
    assert dit.calls[0]['x'].shape[-1] == expected_channels
    assert len(vae.encode_calls) == int(input_image is not None and instruct == 'channel')
    if input_image is None:
        assert torch.count_nonzero(dit.calls[0]['x'][..., 2:]) == 0
    print(label, 'DiT C=', expected_channels,
          'text mode=', text.calls[0]['type_of_content'],
          'VAE encode=', len(vae.encode_calls),
          'text images=', text.calls[0]['images'] is not None)


## 10. `generate_sample_i2v`: изображение уже закодировано

Название аргумента `images` может ввести в заблуждение. Внутри функции **нет** RGB
resize и VAE encode. Внешний I2V pipeline уже подготовил scaled first-frame latent;
здесь ожидается `[1,H,W,C]`, совместимый с `img[:1]`.

Порядок отличается от T2V:

1. `text_embedder.embedder.mode` переключается в `i2v` и **не восстанавливается**.
   Повторное использование этого объекта требует явного выбора режима внешним кодом.
2. RGB не передаётся в `encode`: image conditioning попадает в sampler через `first_frames`.
3. Когда `visual_cond=True`, перед **каждым** шагом `img[:1]` заменяется first-frame latent
   и mask первого латентного кадра становится 1. Conditioning-каналы `visual_cond` остаются
   нулями: в закреплённом коде first-frame information находится в самом `img` и в mask.
4. После последнего Euler шага wrapper снова записывает `latent_visual[:1]=images`.
5. Затем `normalize_first_frame` изменяет первые **четыре** латентных кадра по статистикам
   следующих кадров. Поэтому точное равенство первого латента входному **после wrapper**
   не гарантируется. Это операция коррекции статистик; конкретное намерение автора и требование точного сохранения latent не документированы.
6. Reverse scale → permute → video VAE decode → uint8 выполняются как в T2V.

Мы выбираем `T=6`: нормализатор получает непустую reference-группу и ненулевую source std.
Режимы `T≤4` здесь не считаем исправленными: отдельно исследовали их ограничения выше.

Эти helper edge cases не доказывают отказ I2V 5s (T=31). Отдельная лабораторная 13 проверяет source contracts и реальные пути вызова; число четыре само по себе не классифицируется как ошибка.


In [ ]:
source_excerpt('kandinsky/generation_utils.py', 'def generate_sample_i2v(', 28)
first_latent = torch.linspace(-0.4, 0.4, 8).reshape(1, 2, 2, 2).to(torch.bfloat16)
events_i2v, dit_i2v, vae_i2v, text_i2v = wrapper_components(visual_cond=True)
video_i2v = generation['generate_sample_i2v'](
    video_shape, 'Куб медленно вращается', dit_i2v, vae_i2v, conf_wrapper,
    text_i2v, images=first_latent, num_steps=3, guidance_weight=1.0,
    scheduler_scale=1.0, seed=42, device='cpu', vae_device='cpu',
    progress=False, offload=True, tp_mesh=None,
)
assert video_i2v.shape == (1, 3, 21, 16, 16)
assert len(vae_i2v.encode_calls) == 0  # wrapper не кодирует RGB
assert text_i2v.embedder.mode == 'i2v'
assert all(call['images'] is None for call in text_i2v.calls)
for call in dit_i2v.calls:
    x = call['x']
    assert x.shape == (6, 2, 2, 5)
    assert torch.equal(x[:1, ..., :2], first_latent)
    assert torch.count_nonzero(x[..., 2:4]) == 0
    assert torch.all(x[:1, ..., 4] == 1)
    assert torch.count_nonzero(x[1:, ..., 4]) == 0
decoded_scaled_latents = vae_i2v.decode_calls[0].permute(0, 2, 3, 4, 1)[0] * 0.5
first_change = (decoded_scaled_latents[:1].float() - first_latent.float()).abs().max()
print('Каждый DiT call видит закреплённый first latent и mask=1.')
print('После normalize_first_frame изменение первого latent:', float(first_change))
assert first_change > 0  # normalization действительно меняет точное совпадение


### Опыт: что wrapper делает после последнего шага

Соберём вручную ту же последовательность на маленьких tensors:
сначала восстановим шум по seed, на каждом Euler шаге закрепим первый latent,
после интеграции снова закрепим его, затем вызовем **настоящий** нормализатор.
Сравним результат с тем, что получил fake VAE decoder.

Проверка позволяет отделить три факта: first-frame constraint на входе DiT;
Euler update самого первого кадра после каждого forward; изменение статистик первых
четырёх кадров перед decode. Они происходят в разных местах и имеют разные задачи.


In [ ]:
manual_i2v = torch.randn(6, 2, 2, 2,
    generator=torch.Generator(device='cpu').manual_seed(42), dtype=torch.bfloat16)
for dt in torch.diff(torch.linspace(1, 0, 4)):
    manual_i2v[:1] = first_latent
    manual_i2v += dt * torch.full_like(manual_i2v, 0.25)
manual_i2v[:1] = first_latent
pre_normalization = manual_i2v.clone()
normalized = generation['normalize_first_frame'](manual_i2v)
manual_i2v_decoder = (normalized.reshape(1, 6, 2, 2, 2) / 0.5).permute(0, 4, 1, 2, 3)
assert torch.equal(manual_i2v_decoder, vae_i2v.decode_calls[0])
assert torch.equal(pre_normalization[4:], normalized[4:])
before_mean = pre_normalization.float().mean((1, 2, 3))
after_mean = normalized.float().mean((1, 2, 3))
plt.plot(before_mean.numpy(), 'o-', label='До normalization')
plt.plot(after_mean.numpy(), 's--', label='После normalization')
plt.axvline(3.5, color='gray', linestyle=':', label='Граница первых 4 latent frames')
plt.xlabel('Номер latent frame'); plt.ylabel('Среднее по H,W,C')
plt.legend(); plt.show()
print('Точный путь после sampling воспроизведён; reference frames остались прежними.')


### Итоговая карта wrapper-функций и задания

| Функция | Дополнительный вход | Conditioning в DiT | Encoder VAE внутри | Выход |
|---|---|---|---|---|
| `generate_sample` | нет | zero condition + zero mask при `visual_cond` | нет | video uint8 `[B,3,T_rgb,H_rgb,W_rgb]` |
| `generate_sample_ti2i` | RGB `image` | edit latent + mask при `instruct_type='channel'` | да для channel editing | image/video uint8 по `image_vae` |
| `generate_sample_i2v` | scaled first-frame latent `images` | фиксация noise first frame + mask при `visual_cond` | нет | video uint8 после normalization |

`num_steps`, `guidance_weight`, `scheduler_scale`, `negative_caption`, `offload`
имеют собственные default-значения wrapper-функций. Pipeline передаёт свои настройки;
например Distill 5s должен получать **16 шагов и guidance=1**, а не использовать defaults
`generate_sample` (25 и 5). `progress=False` не отключает `tqdm` в закреплённом `generate`:
аргумент передаётся дальше, но его условие внутри отсутствует.

1. Поставьте `guidance_weight=3`. Сколько DiT calls получится при 3 шагах? Почему text encoder
   calls по-прежнему два? Убедитесь, что постоянная fake velocity не меняется от CFG.
2. Выполните TI2I с `image_vae=False`: объясните дополнительную временную ось входа/выхода.
3. Замените `visual_cond=True` на `False` в I2V и посмотрите сохранённые входы DiT.
   Сам `generate` больше не фиксирует first frame между шагами; wrapper по-прежнему
   восстанавливает его перед normalization. Это разные механизмы.
4. Найдите, где pipeline превращает исходный RGB в аргумент `images` для I2V.
   Почему повторное `encode_video` внутри wrapper нарушило бы договорённость масштабов?
5. Для каждого wrapper запишите, какие данные должны быть на GPU до вызова, какие он
   переносит сам и что сохраняется на CPU после `offload`. Не делайте вывод о памяти
   из вызова `empty_cache`: он не удаляет живые tensors и модели.


## 11. Сводка для Lite Distill 5s и задания

Для 512×768, 5s настоящий wrapper получает shape=(1,31,64,96,16). Patch=(1,2,2) даёт31×32×48=47,616 visual tokens. При visual_cond=True input channels=16+16+1=33, output velocity channels=16. Hunyuan декодирует (31−1)×4+1=121 RGB frames. Для NABLA grid блоков31×4×6=744; mask744², не47,616².

Defaults helper и выбранная модель различаются: generate_sample25 steps/CFG5/scale1; Distill YAML16/1/scale5; T2V pipeline argument default scale10. Записывайте фактически переданные значения. Для w=1 core sampler делает один DiT forward за шаг; wrappers всё равно кодируют negative caption. NFE (number of function evaluations) — число вычислений поля velocity: здесь это число DiT forward. Оно не включает text/VAE расходы.

| Функция | Самая важная проверка |
|---|---|
| get_sparse_params | Spatial block dimensions и temporal patch=1 |
| adaptive_mean_std_normalization | source std не нулевой, reference/statistics определены |
| normalize_first_frame | T≥5 для обычной ветки, Tensor return, первые4frames |
| get_velocity | t×1000 и CFG branch count |
| generate | Inplace img, negative dt, extra channels, seed вне sampler |
| resize_video | Cover + crop, input layout |
| encode_video | Posterior sampling/mode перед scaling для соответствующего codec |
| generate_sample | B=1, noise/text/RoPE/decode, CUDA/offload |
| generate_sample_ti2i | Source image channels+mask и 2D decode |
| generate_sample_i2v | Scaled latent input, persistent mode и normalization после repin |

1. Повторите mask опыт с wT=1 и3. Предскажите изменившиеся блоки.
2. Измените std source/reference и объясните, какие clamp ограничения сработали.
3. Предложите consistent Tensor-returning interface для normalize_first_frame и политику для T≤4. Отдельно обсудите epsilon/std=0. Не меняйте upstream молча.
4. При одинаковом noise сравните w=1 и2, steps4 и8. Разделите NFE, grid, velocity и decoder расходы.
5. В TI2I trace найдите, какие каналы меняет Euler и какие conditioning сохраняются.
6. Для I2V проверьте first frame до/после normalization. Объясните, почему repin перед normalization не гарантирует equality.
7. Составьте план реального CUDA замера: cold/warm, prompt expansion, text residency, peak VRAM/RAM, seed и sampler settings. Заглушки для performance не используйте.

Исходник закреплён в reference/kandinsky-5; все замечания относятся к этой ревизии. Полная генерация pretrained weights здесь не выполняется.


In [ ]:
coverage={n.name for n in function_nodes}
assert coverage==set(gu_cpu['_function_names'])
print('All source functions covered:',len(coverage))
print('Real upstream revision:',REVISION)
print('CPU facade events:',len(generation['_adapter_log']),'global torch.Generator unchanged:',torch.Generator is original_generator)
